# LAB4 — Integração Tavily como Web Search Fallback

---

| Campo | Detalhe |
|---|---|
| **Aula** | Aula 8 — Avaliação e Observabilidade em RAG |
| **Lab** | LAB4 — Tavily Web Search: Integração e Fusão com Docs Locais |
| **Objetivo** | Integrar a API Tavily como web search fallback e formatar resultados para fusão com documentos locais |
| **Duração estimada** | 45–60 minutos |
| **Pré-requisitos** | LAB2 e LAB3 concluídos; chaves Tavily API e OpenAI API válidas |

---

## Contexto Teórico

**Por que precisamos de web search em sistemas RAG jurídicos?**

O direito é uma área altamente dinâmica: novas súmulas são editadas, leis são promulgadas e dados estatísticos são publicados continuamente. Uma base de conhecimento local (vector store) torna-se desatualizada rapidamente.

**Tavily** é uma API de busca na web projetada especificamente para aplicações de IA:
- Retorna conteúdo limpo e estruturado (sem ruído HTML)
- Inclui score de relevância por resultado
- Suporta busca profunda (`search_depth="advanced"`) para fontes especializadas
- Pode incluir a resposta direta do motor de busca (`include_answer=True`)

**Fusão de resultados** (local + web): ao combinar documentos de diferentes fontes, precisamos ponderar e deduplicar para evitar redundância e garantir qualidade.

## 1. Instalação e Configuração das APIs

In [ ]:
# Instalar as bibliotecas necessárias
# tavily-python: cliente oficial da API Tavily
# langchain-community: integrações da comunidade LangChain (inclui TavilySearchResults)
%pip install tavily-python langchain-community langchain langchain-openai -q

In [ ]:
import os
import time
from google.colab import userdata

# Carregar chaves de API do Colab Secrets
# Para adicionar: clique no ícone de chave (🔑) no menu lateral do Colab
try:
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    print("✓ TAVILY_API_KEY carregada")
except Exception:
    os.environ["TAVILY_API_KEY"] = "tvly-..."  # Substitua pela sua chave
    print("⚠ TAVILY_API_KEY definida manualmente")

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✓ OPENAI_API_KEY carregada")
except Exception:
    os.environ["OPENAI_API_KEY"] = "sk-..."  # Substitua pela sua chave
    print("⚠ OPENAI_API_KEY definida manualmente")

print("\nConfiguração concluída.")

## 2. Classe TavilySearchResults — Parâmetros e Configuração

A classe `TavilySearchResults` do LangChain encapsula a API Tavily como uma ferramenta (tool) compatível com agentes LangChain. Principais parâmetros:

| Parâmetro | Tipo | Descrição |
|---|---|---|
| `max_results` | int | Número máximo de resultados retornados (padrão: 5) |
| `search_depth` | str | `"basic"` (rápido) ou `"advanced"` (mais resultados de qualidade, mais lento) |
| `include_answer` | bool | Inclui resposta sintetizada pelo Tavily (True/False) |
| `include_raw_content` | bool | Inclui conteúdo HTML completo da página (True/False) |
| `include_domains` | list | Limitar a domínios específicos (ex: `["jusbrasil.com.br"]`) |
| `exclude_domains` | list | Excluir domínios específicos |

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Configurar a ferramenta Tavily para buscas jurídicas
# search_depth="advanced" para maior qualidade nos resultados
tavily_tool = TavilySearchResults(
    max_results=5,                # Retornar até 5 resultados por busca
    search_depth="advanced",      # Busca aprofundada para conteúdo jurídico
    include_answer=True,          # Incluir resposta sintetizada pelo Tavily
    include_raw_content=False,    # Não incluir HTML completo (economiza tokens)
)

print("✓ TavilySearchResults configurado")
print(f"  max_results: {tavily_tool.max_results}")
print(f"  search_depth: {tavily_tool.search_depth}")
print(f"  include_answer: {tavily_tool.include_answer}")
print(f"  include_raw_content: {tavily_tool.include_raw_content}")

# Também podemos usar o cliente direto para mais controle
from tavily import TavilyClient
cliente_tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
print("\n✓ TavilyClient direto também inicializado")

## 3. Executar 3 Buscas Jurídicas

Demonstraremos buscas em três áreas jurídicas relevantes para o contexto brasileiro contemporâneo.

In [ ]:
# Definir as 3 consultas jurídicas para busca
consultas_juridicas = [
    "novas súmulas STJ 2024 responsabilidade civil",
    "Lei IA Brasil regulamentação 2024",
    "violência doméstica dados estatísticas Brasil 2024"
]

# Dicionário para armazenar os resultados de cada busca
resultados_busca = {}

# Executar cada busca e medir o tempo de resposta
for i, consulta in enumerate(consultas_juridicas, 1):
    print(f"\n{'='*60}")
    print(f"BUSCA {i}: {consulta}")
    print('='*60)
    
    # Medir tempo de resposta da API
    inicio = time.time()
    
    # Invocar a ferramenta Tavily com a consulta
    resultados = tavily_tool.invoke(consulta)
    
    # Calcular latência
    latencia = time.time() - inicio
    
    # Armazenar resultados com metadados
    resultados_busca[consulta] = {
        "resultados": resultados,
        "latencia_s": latencia,
        "num_resultados": len(resultados) if isinstance(resultados, list) else 0
    }
    
    print(f"Tempo de resposta: {latencia:.2f}s")
    print(f"Número de resultados: {len(resultados) if isinstance(resultados, list) else 'N/A'}")
    
    # Exibir prévia do primeiro resultado
    if isinstance(resultados, list) and len(resultados) > 0:
        primeiro = resultados[0]
        print(f"\nPrimeiro resultado:")
        print(f"  URL: {primeiro.get('url', 'N/A')[:80]}")
        conteudo_preview = str(primeiro.get('content', ''))[:200]
        print(f"  Preview: {conteudo_preview}...")
    
    # Pequena pausa para não exceder rate limits
    time.sleep(1)

print("\n✓ Todas as buscas concluídas")

## 4. Estrutura Completa de um Resultado Tavily

Cada resultado Tavily é um dicionário com campos estruturados. Vamos inspecionar todos os campos disponíveis.

In [ ]:
import json

# Selecionar a primeira consulta para análise detalhada
consulta_exemplo = consultas_juridicas[0]
resultados_exemplo = resultados_busca[consulta_exemplo]["resultados"]

print(f"Consulta: '{consulta_exemplo}'")
print(f"Total de resultados: {len(resultados_exemplo)}")
print()

# Inspecionar a estrutura completa do primeiro resultado
if resultados_exemplo:
    primeiro_resultado = resultados_exemplo[0]
    
    print("=" * 60)
    print("ESTRUTURA COMPLETA DO RESULTADO TAVILY")
    print("=" * 60)
    
    # Iterar sobre cada campo do resultado
    for campo, valor in primeiro_resultado.items():
        # Truncar valores longos para exibição
        if isinstance(valor, str) and len(valor) > 200:
            valor_display = valor[:200] + "...[truncado]"
        else:
            valor_display = valor
        
        print(f"\n[{campo}] (tipo: {type(valor).__name__})")
        print(f"  → {valor_display}")
    
    # Explicação de cada campo
    print("\n" + "=" * 60)
    print("GUIA DOS CAMPOS TAVILY")
    print("=" * 60)
    campos_explicacao = {
        "url": "Endereço da página web fonte do resultado",
        "content": "Trecho de texto extraído e limpo (sem HTML)",
        "score": "Relevância estimada pelo Tavily (0-1)",
        "raw_content": "Conteúdo HTML completo (se include_raw_content=True)",
        "title": "Título da página web"
    }
    for campo, explicacao in campos_explicacao.items():
        presente = "✓" if campo in primeiro_resultado else "✗"
        print(f"{presente} {campo:15s}: {explicacao}")

## 5. Função `formatar_resultados_tavily()`

Transformamos os resultados brutos do Tavily em strings formatadas, prontas para uso como contexto em um prompt LLM. A função trunca o conteúdo para controlar o uso de tokens.

In [ ]:
def formatar_resultados_tavily(results: list, max_chars: int = 300) -> list[str]:
    """
    Formata os resultados brutos do Tavily em strings estruturadas.
    
    Args:
        results: Lista de dicionários retornada pela API Tavily
        max_chars: Número máximo de caracteres do conteúdo por resultado
    
    Returns:
        Lista de strings formatadas, prontas para uso como contexto
    """
    # Lista para armazenar os resultados formatados
    docs_formatados = []
    
    for i, resultado in enumerate(results, 1):
        # Extrair campos com fallback para string vazia
        url = resultado.get("url", "URL não disponível")
        titulo = resultado.get("title", "Sem título")
        conteudo = resultado.get("content", "")
        score = resultado.get("score", 0.0)
        
        # Truncar o conteúdo ao limite especificado
        # Truncamento no espaço mais próximo para não cortar palavras
        if len(conteudo) > max_chars:
            conteudo_truncado = conteudo[:max_chars]
            # Encontrar o último espaço antes do limite
            ultimo_espaco = conteudo_truncado.rfind(" ")
            if ultimo_espaco > max_chars * 0.8:  # Se o espaço está perto do limite
                conteudo_truncado = conteudo_truncado[:ultimo_espaco]
            conteudo_final = conteudo_truncado + "..."
        else:
            conteudo_final = conteudo
        
        # Montar a string formatada com metadados úteis
        doc_formatado = (
            f"[Fonte Web {i}] {titulo}\n"
            f"URL: {url}\n"
            f"Relevância Tavily: {score:.2f}\n"
            f"Conteúdo: {conteudo_final}"
        )
        
        docs_formatados.append(doc_formatado)
    
    return docs_formatados


# Testar a função com os resultados da primeira busca
consulta_teste = consultas_juridicas[0]
resultados_brutos = resultados_busca[consulta_teste]["resultados"]

# Formatar com limite de 300 caracteres
docs_formatados = formatar_resultados_tavily(resultados_brutos, max_chars=300)

print(f"Consulta: '{consulta_teste}'")
print(f"Resultados formatados: {len(docs_formatados)}")
print("\n" + "=" * 60)

# Exibir os primeiros 2 resultados formatados
for doc in docs_formatados[:2]:
    print(doc)
    print("-" * 60)

## 6. Função `fundir_com_docs_locais()`

A fusão pondera os documentos de acordo com a confiabilidade relativa de cada fonte:
- **Documentos locais** (peso maior): base de conhecimento curada, validada
- **Resultados web** (peso menor): atuais mas não validados

A ponderação gera um contexto estruturado para o LLM, sinalizando a procedência de cada informação.

In [ ]:
def fundir_com_docs_locais(
    docs_locais: list[str],
    resultados_tavily: list[str],
    pesos: dict = {"local": 0.6, "web": 0.4}
) -> str:
    """
    Cria um contexto ponderado fundindo documentos locais e resultados web.
    
    A fusão organiza o contexto em seções separadas, indicando a fonte e
    o peso relativo de cada conjunto de documentos. Isso permite que o LLM
    dê prioridade adequada às diferentes fontes.
    
    Args:
        docs_locais: Lista de strings com documentos da base local
        resultados_tavily: Lista de strings formatadas pelo formatar_resultados_tavily()
        pesos: Dicionário com pesos relativos de cada fonte
    
    Returns:
        String com contexto ponderado e estruturado
    """
    # Validar que os pesos somam 1.0
    soma_pesos = sum(pesos.values())
    if abs(soma_pesos - 1.0) > 0.01:  # Tolerância de 1%
        print(f"⚠ Aviso: pesos somam {soma_pesos:.2f} (esperado: 1.0)")
    
    # Construir seção de documentos locais
    secao_local = (
        f"=== DOCUMENTOS LOCAIS (peso: {pesos['local']:.0%}) ===\n"
        "[Base de conhecimento interna — alta confiabilidade]\n\n"
    )
    
    # Adicionar cada documento local numerado
    for i, doc in enumerate(docs_locais, 1):
        secao_local += f"[Doc Local {i}]\n{doc}\n\n"
    
    # Construir seção de resultados web
    secao_web = (
        f"=== RESULTADOS WEB — TAVILY (peso: {pesos['web']:.0%}) ===\n"
        "[Fontes da internet — verificar atualidade e autenticidade]\n\n"
    )
    
    # Adicionar cada resultado web
    for resultado in resultados_tavily:
        secao_web += resultado + "\n\n"
    
    # Instrução para o LLM sobre como usar o contexto fundido
    instrucao = (
        "INSTRUÇÃO: Use as informações acima para responder à pergunta. "
        "Priorize os documentos locais (maior peso). "
        "Use os resultados web para complementar com informações mais recentes. "
        "Se houver contradição, sinalize explicitamente.\n"
    )
    
    # Combinar tudo em um contexto único
    contexto_fundido = secao_local + secao_web + instrucao
    
    return contexto_fundido


# Testar a função com documentos de exemplo
docs_locais_exemplo = [
    """O STJ editou a Súmula 637 estabelecendo que é cabível a ação 
    de improbidade administrativa proposta pelo Ministério Público contra 
    os membros do conselho de administração de empresa pública.""",
    
    """A responsabilidade civil objetiva do fornecedor está prevista no 
    art. 12 do Código de Defesa do Consumidor (Lei 8.078/1990)."""
]

# Formatar resultados Tavily
resultados_formatados_teste = formatar_resultados_tavily(
    resultados_busca[consultas_juridicas[0]]["resultados"][:2],
    max_chars=200
)

# Fundir os documentos
contexto = fundir_com_docs_locais(
    docs_locais=docs_locais_exemplo,
    resultados_tavily=resultados_formatados_teste,
    pesos={"local": 0.6, "web": 0.4}
)

print("CONTEXTO FUNDIDO (preview):")
print("-" * 60)
print(contexto[:800] + "...\n[truncado para exibição]")
print(f"\nTotal de caracteres no contexto: {len(contexto):,}")

## 7. Função `deduplicate_results()` — Remoção de Duplicatas

Quando fundimos fontes diferentes, é comum encontrar conteúdo duplicado (o mesmo artigo aparecer na base local e também na web). Usamos **similaridade de strings** (SequenceMatcher) para detectar e eliminar duplicatas, sem necessidade de embeddings.

In [ ]:
from difflib import SequenceMatcher

def deduplicate_results(
    local_docs: list[str],
    web_results: list[str],
    threshold: float = 0.85
) -> tuple[list[str], list[str], list[int]]:
    """
    Remove resultados web que são muito similares a documentos locais.
    
    A deduplicação compara cada resultado web com todos os documentos locais
    usando similaridade de strings (SequenceMatcher). Resultados com similaridade
    acima do threshold são considerados duplicatas e removidos.
    
    Args:
        local_docs: Lista de documentos da base local
        web_results: Lista de resultados web formatados
        threshold: Limiar de similaridade (0-1). Acima desse valor = duplicata
    
    Returns:
        Tupla com (local_docs, web_results_filtrados, indices_removidos)
    """
    # Lista para armazenar índices dos resultados web removidos
    indices_removidos = []
    
    # Lista para armazenar resultados web que não são duplicatas
    web_results_unicos = []
    
    for i, web_result in enumerate(web_results):
        # Verificar similaridade com cada documento local
        eh_duplicata = False
        max_similaridade = 0.0
        
        for local_doc in local_docs:
            # Calcular similaridade usando SequenceMatcher
            # Comparamos versões em lowercase para normalizar
            similaridade = SequenceMatcher(
                None,
                web_result.lower(),  # Resultado web normalizado
                local_doc.lower()    # Documento local normalizado
            ).ratio()
            
            # Guardar a maior similaridade encontrada
            max_similaridade = max(max_similaridade, similaridade)
            
            # Se a similaridade excede o threshold, é duplicata
            if similaridade >= threshold:
                eh_duplicata = True
                break  # Não precisa verificar os demais documentos locais
        
        if eh_duplicata:
            # Registrar como duplicata removida
            indices_removidos.append(i)
            print(f"  Resultado web {i+1} removido (similaridade: {max_similaridade:.2f} ≥ {threshold})")
        else:
            # Manter o resultado único
            web_results_unicos.append(web_result)
    
    return local_docs, web_results_unicos, indices_removidos


# ===== TESTE DA DEDUPLICAÇÃO =====

# Criar um caso de teste com duplicata intencional
docs_locais_teste = [
    "O Superior Tribunal de Justiça editou a Súmula 637 sobre improbidade administrativa.",
    "Responsabilidade civil objetiva do fornecedor — Código de Defesa do Consumidor art. 12."
]

# Resultados web: um é muito similar ao doc local (duplicata), outro é único
web_results_teste = [
    # Duplicata: quase idêntico ao doc local 1
    "[Fonte Web 1] STJ\nConteúdo: O STJ editou Súmula 637 sobre improbidade administrativa e membros do conselho.",
    # Conteúdo único: não existe na base local
    "[Fonte Web 2] JusBrasil\nConteúdo: Nova súmula do STJ 2024 sobre responsabilidade civil em contratos digitais regulamenta...",
    # Outro duplicado parcial
    "[Fonte Web 3] Conjur\nConteúdo: Responsabilidade civil objetiva do fornecedor: art. 12 do CDC aplicado ao e-commerce."
]

print("Executando deduplicação...")
print(f"Documentos locais: {len(docs_locais_teste)}")
print(f"Resultados web antes: {len(web_results_teste)}")
print()

# Executar deduplicação com threshold 0.5 para capturar similares (teste)
docs_locais_final, web_unicos, removidos = deduplicate_results(
    local_docs=docs_locais_teste,
    web_results=web_results_teste,
    threshold=0.5  # Threshold mais baixo para demonstração
)

print(f"\nResultados web após deduplicação: {len(web_unicos)}")
print(f"Removidos: {len(removidos)} (índices: {removidos})")
print("\nResultados únicos mantidos:")
for r in web_unicos:
    print(f"  → {r[:100]}")

## 8. Demo Completo: Pergunta → Tavily + Local → Fusão → Resposta

Demonstração end-to-end do pipeline completo: dado uma pergunta sobre evento recente, buscamos na web (Tavily), fusionamos com docs locais e geramos resposta.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Pergunta sobre evento recente (que provavelmente não está na base local)
PERGUNTA_DEMO = "Quais são as principais disposições da lei brasileira de IA aprovada em 2024?"

print(f"PERGUNTA: {PERGUNTA_DEMO}")
print("=" * 70)

# ---- PASSO 1: Base de conhecimento local (simulada) ----
# Em produção, viria de um vector store (ChromaDB, Pinecone, etc.)
docs_locais_pipeline = [
    """Marco Civil da Internet (Lei 12.965/2014): estabelece princípios, 
    garantias, direitos e deveres para o uso da Internet no Brasil. 
    Prevê neutralidade de rede e proteção à privacidade dos usuários.""",
    
    """Lei Geral de Proteção de Dados - LGPD (Lei 13.709/2018): regula o 
    tratamento de dados pessoais no Brasil, inspirada no GDPR europeu. 
    Criou a ANPD (Autoridade Nacional de Proteção de Dados)."""
]

print(f"\nDocumentos locais carregados: {len(docs_locais_pipeline)}")

# ---- PASSO 2: Busca Tavily ----
print("\nExecutando busca Tavily...")
inicio_tavily = time.time()
resultados_tavily_raw = tavily_tool.invoke(PERGUNTA_DEMO)
latencia_tavily = time.time() - inicio_tavily

print(f"  Tempo Tavily: {latencia_tavily:.2f}s")
print(f"  Resultados obtidos: {len(resultados_tavily_raw) if isinstance(resultados_tavily_raw, list) else 0}")

# ---- PASSO 3: Formatar resultados Tavily ----
if isinstance(resultados_tavily_raw, list):
    docs_web = formatar_resultados_tavily(resultados_tavily_raw, max_chars=300)
else:
    docs_web = []

# ---- PASSO 4: Deduplicar ----
_, docs_web_unicos, _ = deduplicate_results(
    local_docs=docs_locais_pipeline,
    web_results=docs_web,
    threshold=0.85
)
print(f"\nApós deduplicação: {len(docs_web_unicos)} resultados web únicos")

# ---- PASSO 5: Fundir documentos ----
contexto_final = fundir_com_docs_locais(
    docs_locais=docs_locais_pipeline,
    resultados_tavily=docs_web_unicos,
    pesos={"local": 0.6, "web": 0.4}
)

# ---- PASSO 6: Gerar resposta com ChatOpenAI ----
llm_resposta = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", """Você é um especialista em direito digital e tecnologia brasileiro.
Responda a pergunta com base APENAS no contexto fornecido.
Se as informações forem insuficientes, diga explicitamente.
Indique a fonte (local ou web) das informações usadas."""),
    ("human", "CONTEXTO:\n{contexto}\n\nPERGUNTA: {pergunta}")
])

chain_rag = prompt_rag | llm_resposta

print("\nGerando resposta...")
inicio_llm = time.time()
resposta = chain_rag.invoke({"contexto": contexto_final, "pergunta": PERGUNTA_DEMO})
latencia_llm = time.time() - inicio_llm

print("\n" + "=" * 70)
print("RESPOSTA GERADA:")
print("=" * 70)
print(resposta.content)

print("\n" + "=" * 70)
print("RESUMO DO PIPELINE:")
print(f"  Latência Tavily: {latencia_tavily:.2f}s")
print(f"  Latência LLM: {latencia_llm:.2f}s")
print(f"  Latência total: {latencia_tavily + latencia_llm:.2f}s")

## 9. Métricas: Tempo de Resposta e Custo Estimado de Tokens

In [ ]:
import pandas as pd

# ===== COMPARAÇÃO: TAVILY vs RESPOSTA LOCAL =====
# Simular resposta local (sem busca web)

# Prompt apenas com docs locais (sem Tavily)
contexto_local_apenas = fundir_com_docs_locais(
    docs_locais=docs_locais_pipeline,
    resultados_tavily=[],  # Sem resultados web
    pesos={"local": 1.0, "web": 0.0}
)

inicio_local = time.time()
resposta_local = chain_rag.invoke({
    "contexto": contexto_local_apenas,
    "pergunta": PERGUNTA_DEMO
})
latencia_local = time.time() - inicio_local

# ===== CÁLCULO DE TOKENS E CUSTO =====
# Preços gpt-4o-mini (referência: abril/2025)
# Input: $0.15 por 1M tokens | Output: $0.60 por 1M tokens
PRECO_INPUT_POR_1M = 0.15   # USD por 1M tokens de input
PRECO_OUTPUT_POR_1M = 0.60  # USD por 1M tokens de output

def estimar_tokens(texto: str) -> int:
    """Estimativa simples: ~1 token por 4 caracteres (inglês) / 3 chars (português)."""
    return len(texto) // 3

def calcular_custo(tokens_input: int, tokens_output: int) -> float:
    """Calcular custo estimado em USD."""
    custo_input = (tokens_input / 1_000_000) * PRECO_INPUT_POR_1M
    custo_output = (tokens_output / 1_000_000) * PRECO_OUTPUT_POR_1M
    return custo_input + custo_output

# Estimar tokens para cada abordagem
# Abordagem com Tavily
tokens_input_tavily = estimar_tokens(contexto_final + PERGUNTA_DEMO)
tokens_output_tavily = estimar_tokens(resposta.content)
custo_tavily = calcular_custo(tokens_input_tavily, tokens_output_tavily)

# Abordagem local
tokens_input_local = estimar_tokens(contexto_local_apenas + PERGUNTA_DEMO)
tokens_output_local = estimar_tokens(resposta_local.content)
custo_local = calcular_custo(tokens_input_local, tokens_output_local)

# Custo adicional da chamada Tavily (estimativa para plano básico)
# Tavily cobra por busca; plano básico: $0.005 por busca
CUSTO_TAVILY_POR_BUSCA = 0.005

# ===== TABELA COMPARATIVA =====
comparativo = pd.DataFrame([
    {
        "Abordagem": "Local apenas",
        "Latência (s)": f"{latencia_local:.2f}",
        "Tokens Input": tokens_input_local,
        "Tokens Output": tokens_output_local,
        "Custo LLM (USD)": f"${custo_local:.6f}",
        "Custo Tavily (USD)": "$0.000000",
        "Custo Total (USD)": f"${custo_local:.6f}"
    },
    {
        "Abordagem": "Tavily + Local",
        "Latência (s)": f"{latencia_tavily + latencia_llm:.2f}",
        "Tokens Input": tokens_input_tavily,
        "Tokens Output": tokens_output_tavily,
        "Custo LLM (USD)": f"${custo_tavily:.6f}",
        "Custo Tavily (USD)": f"${CUSTO_TAVILY_POR_BUSCA:.6f}",
        "Custo Total (USD)": f"${custo_tavily + CUSTO_TAVILY_POR_BUSCA:.6f}"
    }
])

print("COMPARATIVO: LOCAL vs TAVILY + LOCAL")
print("=" * 80)
print(comparativo.to_string(index=False))

print("\n" + "=" * 80)
print("ANÁLISE DE CUSTO vs BENEFÍCIO:")
print(f"  Custo adicional por usar Tavily: ~${(custo_tavily + CUSTO_TAVILY_POR_BUSCA - custo_local):.6f} USD")
print(f"  Latência adicional: ~{latencia_tavily:.1f}s (busca web)")
print(f"  Benefício: acesso a informações atualizadas sobre eventos recentes")
print(f"\n  Recomendação: use Tavily como FALLBACK (apenas quando docs locais são insuficientes)")

## Conclusão

### O que aprendemos neste lab:

1. **Tavily** oferece resultados estruturados e limpos, ideais para pipelines RAG — sem a necessidade de parsear HTML
2. **Fusão ponderada** permite combinar a confiabilidade da base local com a atualidade da web
3. **Deduplicação** é essencial para evitar redundância que infla o contexto e desperdiça tokens
4. **Trade-off custo vs atualidade**: Tavily adiciona latência e custo, mas é essencial para domínios dinâmicos como o direito

### Perguntas de Reflexão:

1. Em quais situações jurídicas o fallback para web search é **absolutamente necessário**? E quando seria **contraproducente**?

2. Como você garantiria a **confiabilidade** dos resultados Tavily em um contexto jurídico? (Dica: filtragem por domínios como `stj.jus.br`, `planalto.gov.br`)

3. O peso `local: 0.6, web: 0.4` é arbitrário neste lab. Como você calibraria esses pesos de forma **sistemática**?

4. Como a deduplicação baseada em similaridade de strings se compara à deduplicação por **embeddings semânticos**? Quando cada abordagem é preferível?

---
*MBA em RAG & CAG Aplicados a Direito e Segurança Pública · Aula 8*